# AI4MH Governance Layer — Walkthrough
**GSoC 2026 | ISSR, University of Alabama | Mentor: David M. White, MPH, MPA**

End-to-end demonstration of crisis signal scoring and governance across three Alabama counties.
Each county represents a different real-world scenario: genuine crisis, media-driven spike, and rural sparse data.

> **North Star:** A trend intelligence tool for public health decision-makers. No score triggers action autonomously.

## Setup

In [12]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from datetime import datetime, timedelta
import config
from ai4mh.flags import Post, run_all_flags, detect_rural_sparse
from ai4mh.scoring import evaluate_county, EscalationTier, compute_volume_score
from ai4mh.audit import AuditLogger, AnalystAction
from ai4mh.pipeline import _generate_sample_data, COUNTY_METADATA

print('Modules loaded successfully')
print(f'Analysis window : {config.ANALYSIS_WINDOW_HOURS} hours')
print(f'Urban threshold : {config.MIN_POSTS_URBAN} posts')
print(f'Rural threshold : {config.MIN_POSTS_RURAL} posts')
print(f'CSS thresholds  : HIGH={config.CSS_HIGH_THRESHOLD} | MODERATE={config.CSS_MODERATE_THRESHOLD} | LOW={config.CSS_LOW_THRESHOLD}')

Modules loaded successfully
Analysis window : 72 hours
Urban threshold : 30 posts
Rural threshold : 15 posts
CSS thresholds  : HIGH=0.75 | MODERATE=0.5 | LOW=0.3


## 1. Load Sample Data

Three Alabama counties evaluated over a 72-hour window.

In [13]:
posts_by_county = _generate_sample_data()
window_start = datetime.utcnow() - timedelta(hours=config.ANALYSIS_WINDOW_HOURS)

print(f'Window start: {window_start.strftime("%Y-%m-%d %H:%M")} UTC\n')
print(f'{"County":<25} {"FIPS":<8} {"Posts":<8} {"RUCC":<6} {"Type"}')
print('-' * 55)
for fips, posts in posts_by_county.items():
    meta = COUNTY_METADATA.get(fips, {})
    county_type = 'Rural' if meta.get('rucc_code', 1) >= 7 else 'Urban'
    print(f"{meta.get('name','?'):<25} {fips:<8} {len(posts):<8} {meta.get('rucc_code','?'):<6} {county_type}")

Window start: 2026-03-05 10:58 UTC

County                    FIPS     Posts    RUCC   Type
-------------------------------------------------------
Jefferson County          01073    40       1      Urban
Madison County            01089    40       2      Urban
Lowndes County            01085    8        8      Rural


C:\Users\jains\AppData\Local\Temp\ipykernel_23600\1153954988.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  window_start = datetime.utcnow() - timedelta(hours=config.ANALYSIS_WINDOW_HOURS)


## 2. Sample Posts

A quick look at the language patterns in each county.

In [14]:
for fips, posts in posts_by_county.items():
    meta = COUNTY_METADATA.get(fips, {})
    print(f"\n{meta.get('name')} ({fips})")
    print('-' * 45)
    for p in posts[:3]:
        text = p.text[:85] + '...' if len(p.text) > 85 else p.text
        print(f'  {text}')


Jefferson County (01073)
---------------------------------------------
  I can't do this anymore. Everything feels hopeless and I don't see a way out.
  Been having really dark thoughts lately. Can't sleep, can't eat. I'm exhausted.
  I don't want to be here anymore. The pain is too much.

Madison County (01089)
---------------------------------------------
  Did you hear about the suicide case reported in the news today? So sad.
  Breaking news: mental health crisis reported locally. Have you seen this?
  According to the article, there's been an increase in mental health issues.

Lowndes County (01085)
---------------------------------------------
  I'm not doing well. Really struggling out here and there's nowhere to go.
  Feeling real low. Nothing seems worth it anymore.
  Been a hard week. Can't shake this feeling of hopelessness.


## 3. Governance Flag Detection

Flags modify signal reliability before scoring. They never suppress a signal — they discount volume and surface a plain-language note to the reviewer.

In [15]:
flag_results = {}

for fips, posts in posts_by_county.items():
    meta = COUNTY_METADATA.get(fips, {})
    flag_result = run_all_flags(
        posts=posts, county_fips=fips,
        rucc_code=meta['rucc_code'], window_start=window_start
    )
    flag_results[fips] = flag_result

    print(f"\n{meta.get('name')} ({fips})")
    print('-' * 45)
    if flag_result.active_flags:
        print(f"  Flags   : {', '.join(flag_result.active_flags)}")
        print(f"  Discount: {flag_result.volume_discount:.0%}")
        for note in flag_result.plain_language_notes:
            print(f"  Note    : {note}")
    else:
        print('  No flags — signal unmodified')


Jefferson County (01073)
---------------------------------------------
  Flags   : bot_risk
  Discount: 100%
  Note    : Near-duplicate clustering: 1 posts identified as coordinated (high text similarity within 6 hours). Cluster contributes 1 post-weight to volume.

Madison County (01089)
---------------------------------------------
  Flags   : bot_risk, media_context
  Discount: 60%
  Note    : Near-duplicate clustering: 8 posts identified as coordinated (high text similarity within 6 hours). Cluster contributes 1 post-weight to volume.
  Note    : 100% of posts classified as reportative (discussing news rather than personal distress). Volume component discounted to reflect genuine distress signal.

Lowndes County (01085)
---------------------------------------------
  Flags   : rural_sparse, rural_bias
  Discount: 100%
  Note    : Sample size below threshold (8 posts, minimum 15 for rural county). Signal routed to sparse-data review queue. Treat confidence estimate with additional 

> Jefferson County: no flags, signal passes through at full weight. 
> Madison County: bot + media flags active, volume discounted 50%. 
> Lowndes County: rural sparse + rural bias flags — will be routed to human review queue.

## 4. Conditional CSS Scoring

Sentiment anchors the score. Volume and geography modify confidence. If sentiment is extreme (> 0.85), it stands alone — no corroboration needed.

In [16]:
css_results = {}

for fips, posts in posts_by_county.items():
    meta = COUNTY_METADATA.get(fips, {})
    result = evaluate_county(
        posts=posts, county_fips=fips, rucc_code=meta['rucc_code'],
        window_start=window_start, baseline_mean=meta['baseline_mean'],
        baseline_std=meta['baseline_std'], flag_result=flag_results[fips]
    )
    css_results[fips] = result

    print(f"\n{meta.get('name')} ({fips})")
    print('-' * 45)
    if result.routed_to_sparse_queue:
        print(f'  Posts : {result.n_posts} — routed to sparse-data review queue')
    else:
        print(f'  Sentiment : {result.components.sentiment:.3f}')
        print(f'  Volume    : {result.components.volume:.3f} → adjusted {result.components.volume_adjusted:.3f}')
        print(f'  Geography : {result.components.geography:.3f}')
        print(f'  CSS Score : {result.css:.3f}  [{result.escalation_tier.value}]')
    print(f'  Confidence: {result.confidence.percentage:.0%} — {result.confidence.plain_language}')


Jefferson County (01073)
---------------------------------------------
  Sentiment : 0.284
  Volume    : 0.646 → adjusted 0.646
  Geography : 0.000
  CSS Score : 0.354  [LOW]
  Confidence: 10% — Signal driven by significant volume spike — Caution: flags active: bot_risk.

Madison County (01089)
---------------------------------------------
  Sentiment : 0.007
  Volume    : 0.889 → adjusted 0.533
  Geography : 0.000
  CSS Score : 0.190  [NOISE]
  Confidence: 10% — Signal driven by weak signal across components — Caution: flags active: bot_risk, media_context.

Lowndes County (01085)
---------------------------------------------
  Posts : 8 — routed to sparse-data review queue
  Confidence: 20% — Insufficient data (8 posts, minimum 15 for rural county). Routed to sparse-data review queue. Apply significant additional judgment.


> Jefferson County scores LOW — genuine signal but below escalation threshold. 
> Madison County scores NOISE — volume discounted by flags, signal unreliable. 
> Lowndes County routes to sparse queue — 8 posts below rural minimum of 15.

## 5. Escalation Summary

All counties classified and routed. No action is taken without human confirmation.

In [17]:
tier_icons = {'HIGH': '[HIGH]', 'MODERATE': '[MODERATE]', 'LOW': '[LOW]', 'NOISE': '[NOISE]'}

print(f'{"County":<25} {"CSS":<8} {"Tier":<12} {"Confidence":<12} {"Flags"}')
print('-' * 70)
for fips, result in css_results.items():
    meta = COUNTY_METADATA.get(fips, {})
    tier = tier_icons.get(result.escalation_tier.value, '?')
    flags_str = ', '.join(result.flags.active_flags) if result.flags.active_flags else 'none'
    sparse = ' (sparse)' if result.routed_to_sparse_queue else ''
    print(f"{meta.get('name','?'):<25} {result.css:<8.3f} {tier:<12} {result.confidence.percentage:<12.0%} {flags_str}{sparse}")

print()
print('No autonomous action taken. Human confirmation required before any intervention.')

County                    CSS      Tier         Confidence   Flags
----------------------------------------------------------------------
Jefferson County          0.354    [LOW]        10%          bot_risk
Madison County            0.190    [NOISE]      10%          bot_risk, media_context
Lowndes County            0.000    [LOW]        20%          rural_sparse, rural_bias (sparse)

No autonomous action taken. Human confirmation required before any intervention.


## 6. Audit Logging

Every evaluation generates an immutable record — four categories: what the system **saw**, what it **decided**, what the **human did**, what **happened after**.

In [18]:
import tempfile, os

tmp_log = tempfile.mktemp(suffix='.jsonl')
logger = AuditLogger(tmp_log)

audit_records = {}
for fips, result in css_results.items():
    record = logger.log_evaluation(result)
    audit_records[fips] = record

# Show Jefferson County record
rec = audit_records['01073'].to_dict()
print('Sample audit record — Jefferson County')
print('-' * 45)
for field in ['event_id', 'timestamp_utc', 'county_fips', 'n_posts',
              'css', 'escalation_tier', 'active_flags',
              'confidence_pct', 'confidence_plain_language']:
    print(f'  {field:<35} {rec[field]}')
print()
print('Human review fields (populated when analyst reviews):')
print(f'  analyst_id      : {rec["analyst_id"]}')
print(f'  analyst_action  : {rec["analyst_action"]}')

Sample audit record — Jefferson County
---------------------------------------------
  event_id                            a923a459-d07d-4c97-94d9-5dc1e34fbe7e
  timestamp_utc                       2026-03-08T10:58:54.382111Z
  county_fips                         01073
  n_posts                             40
  css                                 0.3541
  escalation_tier                     LOW
  active_flags                        ['bot_risk']
  confidence_pct                      0.1
  confidence_plain_language           Signal driven by significant volume spike — Caution: flags active: bot_risk.

Human review fields (populated when analyst reviews):
  analyst_id      : None
  analyst_action  : None


## 7. Human Review

The analyst reviews the Jefferson County signal and logs their decision. No CSS score triggers action without this step.

In [19]:
result = css_results['01073']

print('Jefferson County — Analyst Review')
print('-' * 45)
print(f'CSS Score  : {result.css:.3f}')
print(f'Tier       : {result.escalation_tier.value}')
print(f'Confidence : {result.confidence.percentage:.0%}')
print(f'Flags      : {result.flags.active_flags or "none"}')
print(f'Note       : {result.confidence.plain_language}')
print()

success = logger.log_analyst_review(
    event_id=audit_records['01073'].event_id,
    analyst_id='analyst_dmw_001',
    action=AnalystAction.CONFIRMED,
    rationale=(
        'First-person distress expressions across 25+ unique accounts. '
        'Patterns consistent with genuine community distress, not media-reactive. '
        'Recommending outreach to county behavioral health coordinator.'
    )
)

print(f'Review logged: {"Success" if success else "Failed"}')
print()
print('Next step: County behavioral health coordinator notified for human decision on intervention.')

Jefferson County — Analyst Review
---------------------------------------------
CSS Score  : 0.354
Tier       : LOW
Confidence : 10%
Flags      : ['bot_risk']
Note       : Signal driven by significant volume spike — Caution: flags active: bot_risk.

Review logged: Success

Next step: County behavioral health coordinator notified for human decision on intervention.


## 8. Flag Impact on Volume

The same post count is interpreted differently depending on which flags are active.

In [20]:
baseline_mean, baseline_std, n_posts = 20, 8, 60
v_raw = compute_volume_score(n_posts, baseline_mean, baseline_std)

print('Scenario: 60 posts | baseline mean=20 | std=8')
print()
print(f'{"Scenario":<28} {"Raw":<10} {"Adjusted":<10} {"Reduction"}')
print('-' * 55)
for scenario, discount in [
    ('No flags',            1.0),
    ('Bot risk',            0.5),
    ('Media spike',         0.6),
    ('Bot + media',         0.5),
]:
    adj = round(v_raw * discount, 3)
    red = f'{(1-discount)*100:.0f}%' if discount < 1.0 else 'none'
    print(f'{scenario:<28} {v_raw:<10.3f} {adj:<10.3f} {red}')

print()
print('Volume is discounted, not deleted. The reason is always surfaced to the reviewer.')

Scenario: 60 posts | baseline mean=20 | std=8

Scenario                     Raw        Adjusted   Reduction
-------------------------------------------------------
No flags                     1.000      1.000      none
Bot risk                     1.000      0.500      50%
Media spike                  1.000      0.600      40%
Bot + media                  1.000      0.500      50%

Volume is discounted, not deleted. The reason is always surfaced to the reviewer.


## 9. Rural vs Urban Thresholds

Rural counties use a lower minimum threshold — reflecting lower social media activity, not lower crisis risk.

In [21]:
print(f'Urban minimum: {config.MIN_POSTS_URBAN} posts | Rural minimum: {config.MIN_POSTS_RURAL} posts')
print()
print(f'{"Scenario":<38} {"Flags":<28} {"Action"}')
print('-' * 80)
for name, fips, n, rucc in [
    ('Jefferson Co (urban, 8 posts)',   '01073', 8,  2),
    ('Jefferson Co (urban, 35 posts)',  '01073', 35, 2),
    ('Lowndes Co (rural, 8 posts)',     '01085', 8,  8),
    ('Lowndes Co (rural, 20 posts)',    '01085', 20, 8),
]:
    result = detect_rural_sparse(fips, n, rucc)
    flags = ', '.join(result.active_flags) if result.active_flags else 'none'
    action = 'sparse queue' if 'rural_sparse' in result.active_flags else 'proceeds to scoring'
    print(f'{name:<38} {flags:<28} {action}')

print()
print('Rural counties below threshold → sparse queue (human review), never silence.')

Urban minimum: 30 posts | Rural minimum: 15 posts

Scenario                               Flags                        Action
--------------------------------------------------------------------------------
Jefferson Co (urban, 8 posts)          rural_sparse                 sparse queue
Jefferson Co (urban, 35 posts)         none                         proceeds to scoring
Lowndes Co (rural, 8 posts)            rural_sparse, rural_bias     sparse queue
Lowndes Co (rural, 20 posts)           rural_bias                   proceeds to scoring

Rural counties below threshold → sparse queue (human review), never silence.


## 10. Full Audit Trail

Every record written during this session — showing the complete provenance chain.

In [22]:
all_records = logger.get_all_records()
print(f'Total records: {len(all_records)}')
print()

for i, rec in enumerate(all_records, 1):
    rtype = rec.get('record_type', 'evaluation')
    print(f'Record {i}: {rtype.upper()}')
    if rtype == 'evaluation':
        print(f"  County: {rec.get('county_fips')} | CSS: {rec.get('css')} | Tier: {rec.get('escalation_tier')} | Flags: {rec.get('active_flags')}")
    elif rtype == 'analyst_review':
        print(f"  Event: {rec.get('event_id','')[:8]}... | Action: {rec.get('analyst_action')} | Analyst: {rec.get('analyst_id')}")
    print()

print('Records are append-only. In production: retained 7 years, tamper-evident storage.')
os.remove(tmp_log)

Total records: 4

Record 1: EVALUATION
  County: 01073 | CSS: 0.3541 | Tier: LOW | Flags: ['bot_risk']

Record 2: EVALUATION
  County: 01089 | CSS: 0.19 | Tier: NOISE | Flags: ['bot_risk', 'media_context']

Record 3: EVALUATION
  County: 01085 | CSS: 0.0 | Tier: LOW | Flags: ['rural_sparse', 'rural_bias']

Record 4: ANALYST_REVIEW
  Event: a923a459... | Action: CONFIRMED | Analyst: analyst_dmw_001

Records are append-only. In production: retained 7 years, tamper-evident storage.


## Summary

| Step | Component | Key Decision |
|------|-----------|-------------|
| 1–2 | Data loading | 72-hour window, 3 county scenarios |
| 3 | Governance flags | Discount + flag, never delete |
| 4 | CSS scoring | Sentiment anchors; volume/geo modify confidence |
| 5 | Escalation | Tiered routing, no autonomous action |
| 6 | Audit logging | 4 categories: saw / decided / human did / outcome |
| 7 | Human review | Mandatory before any intervention |
| 8 | Flag impact | Volume discounted but always visible to reviewer |
| 9 | Rural equity | Tiered thresholds — rural never silenced |
| 10 | Audit trail | Every decision reconstructable |

**GitHub:** github.com/Swasti2004/gsoc2026-ai4mh-governance